# 05 - Working with Files

Your capstone bot can read photos and PDFs people send it, and hand back files it builds itself (like a spreadsheet from `code_interpreter`). This notebook covers both directions using the exact same patterns `app.py` uses -- by the end, the file-handling code in the capstone repo won't look like a black box.

Two directions:
1. **Files going in** -- giving the agent an image or document to look at.
2. **Files coming out** -- the agent building a real file and handing it back to you.


In [ ]:
%pip install -q openai-agents httpx

import os

# Don't share or commit this notebook with your key filled in.
os.environ["OPENAI_API_KEY"] = "sk-..."  # <-- paste your key here

print("Ready to go.")

## 1. Files going in -- the simple way

If a file is already sitting at a public URL, you don't need to upload anything. Just point `input_image` at it directly.


In [ ]:
from agents import Agent, Runner

agent = Agent(name="Image Describer", instructions="Describe what you see, briefly.")

result = Runner.run_sync(
    agent,
    [{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "What does this image show?"},
            {"type": "input_image", "image_url": "https://httpbin.org/image/png", "detail": "auto"},
        ],
    }],
)
print(result.final_output)

## 2. Files going in -- the upload way

Telegram photos aren't at a public URL you control -- they're behind Telegram's own API, gated by your bot token. So the real bot can't just pass a URL; it has to **download the bytes, then upload them to OpenAI's Files API**, and reference the result by `file_id`. This is exactly what `_build_input_content()` in `app.py` does.

Let's do the same thing here: download a test image, then upload it ourselves.


In [ ]:
import httpx
from openai import OpenAI

openai_client = OpenAI()

# Stand-in for "download the photo bytes from Telegram"
image_bytes = httpx.get("https://httpbin.org/image/png").content

# Upload to OpenAI -- purpose="vision" for images
uploaded = openai_client.files.create(file=("photo.png", image_bytes), purpose="vision")
print("Uploaded as:", uploaded.id)

In [ ]:
result = Runner.run_sync(
    agent,
    [{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "What does this image show?"},
            {"type": "input_image", "file_id": uploaded.id, "detail": "auto"},
        ],
    }],
)
print(result.final_output)

## 3. Documents (PDFs) work the same way

Same upload pattern, different `purpose` (`"user_data"` instead of `"vision"`) and content type (`input_file` instead of `input_image`).


In [ ]:
pdf_bytes = httpx.get("https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf").content

uploaded_pdf = openai_client.files.create(file=("document.pdf", pdf_bytes), purpose="user_data")

result = Runner.run_sync(
    agent,
    [{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "What is this document about?"},
            {"type": "input_file", "file_id": uploaded_pdf.id, "filename": "document.pdf"},
        ],
    }],
)
print(result.final_output)

### Why `purpose` matters

`"vision"` and `"user_data"` aren't arbitrary labels -- they tell OpenAI *how* to process the file before the model sees it. Use `"vision"` for anything the model should look at as an image, and `"user_data"` for documents it should read.


## 4. Files coming out -- the agent builds one for you

This is the other direction: instead of giving the agent a file, let it build one. `code_interpreter` (from notebook 04) can write real files -- a `.xlsx`, a `.pptx`, whatever the task calls for. The tricky part isn't asking for the file, it's **getting it back out** of OpenAI's sandbox.


In [ ]:
from agents import CodeInterpreterTool

file_agent = Agent(
    name="File Builder",
    instructions="Use code_interpreter to build real files with the data given, don't just describe it.",
    tools=[
        CodeInterpreterTool(tool_config={"type": "code_interpreter", "container": {"type": "auto"}}),
    ],
)

result = Runner.run_sync(
    file_agent,
    "Build a small .xlsx with two columns: Fruit (apple, banana, cherry) and Price (1.20, 0.50, 3.00).",
)
print(result.final_output)

### Finding the file the agent built

When `code_interpreter` creates a file, the model's reply carries a `container_file_citation` annotation pointing at it -- that's the signal to look for (there's no need to guess or list everything in the sandbox). This is exactly what `_extract_generated_files()` in `app.py` does.


In [ ]:
def find_generated_files(result):
    """Return (container_id, file_id, filename) for every file the agent created."""
    files = []
    for item in result.new_items:
        if item.type != "message_output_item":
            continue
        for content in item.raw_item.content:
            for annotation in getattr(content, "annotations", None) or []:
                if getattr(annotation, "type", None) == "container_file_citation":
                    files.append((annotation.container_id, annotation.file_id, annotation.filename))
    return files


generated = find_generated_files(result)
print(generated)

### Downloading it

Files `code_interpreter` creates live in a **container**, not the regular Files API -- so downloading them uses a different endpoint: `client.containers.files.content`. Your bot forwards these bytes straight to Telegram as a document; here, we'll just save it to disk to prove it works.


In [ ]:
if generated:
    container_id, file_id, filename = generated[0]
    file_content = openai_client.containers.files.content.retrieve(file_id, container_id=container_id)

    with open(filename, "wb") as f:
        f.write(file_content.read())

    print(f"Saved {filename} -- open it and check the data made it in correctly.")
else:
    print("No file was generated -- try re-running the cell above, or rephrase the request.")

## Recap

| What you did | Where it lives in the capstone repo |
|---|---|
| Upload a photo/PDF, reference by `file_id` | `app.py` -- `_build_input_content()` |
| Find a file the agent generated | `app.py` -- `_extract_generated_files()` |
| Download it from the container | `app.py` -- the `containers.files.content.retrieve(...)` call in `/process` |

The only difference in the real bot: instead of saving to disk, the bytes get forwarded straight to Telegram as a document.

**Next:** open `06_multi_turn_conversations.ipynb`.
